Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Вопрос</td><td>Ответ</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [2]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [13]:
texts = [
    'выиграйте бесплатно приз деньги',
    'бесплатно деньги переведите срочно',
    'выиграйте деньги сейчас бесплатно',
    'срочно переведите деньги приз',
    'бесплатно приз выиграйте сейчас',
    'деньги бесплатно срочно приз',
    'получите бесплатно крупный приз',
    'переведите деньги немедленно бесплатно',
    'встреча в офисе завтра утром',
    'проект отчет встреча команда',
    'обсудим проект на встрече завтра',
    'команда офис встреча сегодня',
    'отчет по проекту готов завтра',
    'встреча команды в офисе утром',
    'планируем встречу с командой завтра',
    'отправьте отчет по проекту сегодня',
]
labels = [1]*8 + [0]*8     # 1 = спам, 0 = не спам
 
# Разбиваем на обучающую и тестовую выборки (75% / 25%)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels)
 
print(f"Обучающих примеров: {len(X_train_text)}, тестовых: {len(X_test_text)}")
 
# Векторизация: превращаем текст в "мешок слов"
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train_text)     # обучите векторизатор и преобразуйте обучающие тексты
 
print("Размер словаря:", len(vectorizer.vocabulary_))
print("Слова в словаре:", vectorizer.get_feature_names_out())


Обучающих примеров: 12, тестовых: 4
Размер словаря: 24
Слова в словаре: ['бесплатно' 'встреча' 'встрече' 'выиграйте' 'готов' 'деньги' 'завтра'
 'команда' 'команды' 'на' 'немедленно' 'обсудим' 'отправьте' 'отчет'
 'офисе' 'переведите' 'по' 'приз' 'проект' 'проекту' 'сегодня' 'сейчас'
 'срочно' 'утром']


<table>
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Посмотрите на список слов в словаре. <br>
    Какие из них, по вашей интуиции, скорее всего окажутся наиболее характерными для класса «спам»,<br>
    а какие — для класса «не спам»? Сравните свою интуицию с результатами задания 4</td><td>Слова [бесплатно, выиграйте, отправьте] кажутся странными</td></tr>
</table>

In [14]:
# Преобразуем тестовые тексты тем же словарём (НЕ обучаем заново!)
X_test = vectorizer.transform(X_test_text)

# Создаём и обучаем модель
model = MultinomialNB()
model.fit(X_train, y_train)

# Априорные вероятности классов (в логарифмах, поэтому применяем exp)
class_priors = np.exp(model.class_log_prior_)
for cls, prior in zip(model.classes_, class_priors):
    label_name = "спам" if cls == 1 else "не спам"
    print(f"P({label_name}) = {prior:.4f}")


P(не спам) = 0.5000
P(спам) = 0.5000



<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Совпадают ли выведенные моделью априорные вероятности с тем, <br>
    что мы ожидали (0,5 для каждого класса, <br>
    поскольку в обучающей выборке классы сбалансированы)?</td><td>Да, совпадают</td></tr>
</table>

In [15]:
predictions = model.predict(X_test)          # получите предсказания на тестовой выборке
accuracy = accuracy_score(predictions, y_test)      # сравните истинные и предсказанные метки
 
print(f"Точность (accuracy) на тестовой выборке: {accuracy:.2%}")
print()
 
probs = model.predict_proba(X_test)
for text, true_label, pred_label, prob in zip(X_test_text, y_test, predictions, probs):
    label_name = lambda l: "спам" if l == 1 else "не спам"
    print(f"Текст: «{text}»")
    print(f"  Истинная метка: {label_name(true_label)}, Предсказано: {label_name(pred_label)}, P(спам)={prob[1]:.4f}")
    print()
 
# Классификация собственных примеров
my_texts = [
    'бесплатно деньги приз',
    'встреча завтра в офисе',
    "Получите деньги за так"
]
my_vectors = vectorizer.transform(my_texts)
my_predictions = model.predict(my_vectors)
my_probs = model.predict_proba(my_vectors)
 
for text, pred, prob in zip(my_texts, my_predictions, my_probs):
    label_name = "спам" if pred == 1 else "не спам"
    print(f"«{text}» → {label_name} (P(спам)={prob[1]:.4f})")


Точность (accuracy) на тестовой выборке: 100.00%

Текст: «команда офис встреча сегодня»
  Истинная метка: не спам, Предсказано: не спам, P(спам)=0.0697

Текст: «бесплатно деньги переведите срочно»
  Истинная метка: спам, Предсказано: спам, P(спам)=0.9976

Текст: «планируем встречу с командой завтра»
  Истинная метка: не спам, Предсказано: не спам, P(спам)=0.2099

Текст: «получите бесплатно крупный приз»
  Истинная метка: спам, Предсказано: спам, P(спам)=0.9713

«бесплатно деньги приз» → спам (P(спам)=0.9954)
«встреча завтра в офисе» → не спам (P(спам)=0.0244)
«Получите деньги за так» → спам (P(спам)=0.8644)


<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Правильно ли модель классифицировала все тестовые письма?<br>
    Совпал ли результат для ваших собственных придуманных писем с вашими ожиданиями? <br>
    Если нет — можете предположить, почему?</td><td>Всё чётко</td></tr>
</table>

In [17]:
feature_names = vectorizer.get_feature_names_out()
log_prob = model.feature_log_prob_     # форма (число классов, число слов)
 
# Разница log P(слово|спам) - log P(слово|не спам)
# Внимание: model.classes_ подскажет, в какой строке какой класс
diff = log_prob[1] - log_prob[0]     # укажите индекс строки для класса "не спам"
 
order = np.argsort(diff)[::-1]         # сортировка по убыванию
 
print("Топ-5 слов, характерных для СПАМА:")
for i in order[:5]:
    print(f"  {feature_names[i]}: разница = {diff[i]:.3f}")
 
print()
print("Топ-5 слов, характерных для НЕ СПАМА:")
for i in order[-5:]:
    print(f"  {feature_names[i]}: разница = {diff[i]:.3f}")


Топ-5 слов, характерных для СПАМА:
  деньги: разница = 1.852
  бесплатно: разница = 1.852
  приз: разница = 1.670
  выиграйте: разница = 1.447
  переведите: разница = 1.159

Топ-5 слов, характерных для НЕ СПАМА:
  утром: разница = -1.038
  офисе: разница = -1.038
  отчет: разница = -1.326
  завтра: разница = -1.326
  встреча: разница = -1.326


<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Совпал ли список наиболее характерных слов с вашей интуицией из задания 1? <br>
    Свяжите этот результат с примером из лекции: слова «бесплатно» и «деньги» должны <br>
    получить высокую положительную разницу (характерны для спама), а «встреча» и «отчет»<br>
     — отрицательную (характерны для не-спама).</td><td>Да, всё примерно так и есть</td></tr>
</table>

In [19]:
from sklearn.naive_bayes import BernoulliNB
 
model_bernoulli = BernoulliNB()
model_bernoulli.fit(X_train, y_train)
 
predictions_bernoulli = model_bernoulli.predict(X_test)
accuracy_bernoulli = accuracy_score(y_test, predictions_bernoulli)
 
print(f"Точность MultinomialNB: {accuracy:.2%}")
print(f"Точность BernoulliNB:   {accuracy_bernoulli:.2%}")


Точность MultinomialNB: 100.00%
Точность BernoulliNB:   100.00%


<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>На таком маленьком наборе данных с короткими письмами <br>
    результаты MultinomialNB и BernoulliNB, скорее всего, окажутся<br>
     похожими или одинаковыми. Как вы думаете, в каких случаях <br>
     (при каком типе текстов) разница между ними могла бы стать более заметной?<br>
      Подсказка: подумайте о письмах, где одно и то же слово «бесплатно» повторяется много раз.</td><td>На больших тестах результат может отличаться. Слова могут входить в предложения несколько раз</td></tr>
</table>